In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import shutil

# ==============================
# 1. KONFIGURASI
# ==============================
DATA_DIR     = 'data'
QC_DIR       = os.path.join(DATA_DIR,'01.QC_Dataset_Level_01')
REGIONAL_DIR = os.path.join(DATA_DIR,'02.Regionalisasi_Dataset')

if os.path.exists(REGIONAL_DIR):
    shutil.rmtree(REGIONAL_DIR)
os.makedirs(REGIONAL_DIR, exist_ok=True)

# Daftar baseline yang akan diproses
BASELINES = ['1981', '1991']

# Buat direktori utama
os.makedirs(REGIONAL_DIR, exist_ok=True)

# Konfigurasi per parameter
PARAM_CONFIG = {
    'TEMPERATURE_AVG_C': {
        'n_clusters': 4,
        'random_state': 42,
        'description': 'Suhu rata-rata harian'
    },
    'TEMP_24H_TN_C': {
        'n_clusters': 4,
        'random_state': 42,
        'description': 'Suhu minimum harian'
    },
    'TEMP_24H_TX_C': {
        'n_clusters': 4,
        'random_state': 42,
        'description': 'Suhu maksimum harian'
    },
    'RAINFALL_24H_MM': {
        'n_clusters': 6,
        'random_state': 42,
        'description': 'Curah hujan harian'
    }
}

# ==============================
# 2. FUNGSI PROSES PER PARAMETER & BASELINE
# ==============================
def process_parameter(param, config, baseline):
    print(f"\n{'='*60}")
    print(f"Memproses: {param} | Baseline: {baseline}")
    print(f"Deskripsi: {config['description']}")
    print(f"Jumlah cluster: {config['n_clusters']}")
    print(f"{'='*60}")
    
    # Path input dari pipeline QC
    if param == 'RAINFALL_24H_MM':
        input_file = os.path.join(
            QC_DIR, param, '06.Adjusted',
            f'{param}_FINAL_QC_DATA_LEVEL1_80PCT_ONLY_{baseline}.csv'
            #f'06.rainfall_extended_{baseline}.csv'
            #f'05.abrupt_adjusted_{baseline}.csv'
        )
    else:
        input_file = os.path.join(
            QC_DIR, param, '06.Adjusted',
            #f'05.abrupt_adjusted_{baseline}.csv'
            f'{param}_FINAL_QC_DATA_LEVEL1_80PCT_ONLY_{baseline}.csv'
        )

    # Direktori output spesifik per parameter + baseline
    output_dir = os.path.join(REGIONAL_DIR, f"{param}_BASELINE_{baseline}")
    os.makedirs(os.path.join(output_dir, 'plots'), exist_ok=True)
    if not os.path.exists(input_file):
        print(f"⚠️  File tidak ditemukan: {input_file}. Lewati.")
        return False
    try:
        df = pd.read_csv(input_file, parse_dates=['DATA_TIMESTAMP'])
        if param == 'RAINFALL_24H_MM':
            qc_col = f'QC_RAINFALL_24H_MM_ROBI_EXTEND'
        else:
            qc_col = f'QC_{param}'
        
        if qc_col not in df.columns:
            print(f"⚠️  Kolom '{qc_col}' tidak ditemukan. Kolom tersedia: {list(df.columns)}")
            return False
        # Agregasi per stasiun
        agg_df = df.groupby('WMO_ID').agg({
            'CURRENT_LATITUDE': 'first',
            'CURRENT_LONGITUDE': 'first',
            qc_col: 'mean'
        }).reset_index()
        agg_df = agg_df.dropna(subset=[qc_col]).copy()
        print(f"Jumlah stasiun valid: {len(agg_df)}")
        if len(agg_df) == 0:
            print("⚠️  Tidak ada stasiun dengan data valid. Lewati.")
            return False
        # PCA
        X = agg_df[['CURRENT_LATITUDE', 'CURRENT_LONGITUDE', qc_col]].values
        pca_full = PCA()
        pca_full.fit(X)
        explained_var = pca_full.explained_variance_ratio_
        cumulative_var = np.cumsum(explained_var)
        # Simpan konfigurasi
        with open(os.path.join(output_dir, 'pca_kmeans_config.txt'), 'w') as f:
            f.write(f"Parameter: {param}\n")
            f.write(f"Baseline: {baseline}\n")
            f.write(f"Deskripsi: {config['description']}\n")
            f.write(f"Jumlah stasiun: {len(agg_df)}\n")
            f.write(f"Jumlah cluster: {config['n_clusters']}\n")
            f.write(f"Random state: {config['random_state']}\n")
            f.write("\nExplained Variance per Komponen:\n")
            for i, (var, cum) in enumerate(zip(explained_var, cumulative_var)):
                line = f"Komponen {i+1}: Variansi = {var:.4f}, Kumulatif = {cum:.4f}\n"
                f.write(line)
                print(line.strip())
        # Plot variansi kumulatif
        plt.figure(figsize=(6, 4))
        plt.plot(range(1, len(explained_var)+1), cumulative_var, marker='o', color='steelblue')
        plt.xlabel('Jumlah Komponen')
        plt.ylabel('Kumulatif Variansi')
        plt.title('PCA: Explained Variance Cumulative')
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'plots', 'Explained_Variance_Cumulative.png'), dpi=300, bbox_inches='tight')
        plt.close()
        
        # Clustering
        pca = PCA(n_components=2)
        X_pca = pca.fit_transform(X)
        
        kmeans = KMeans(
            n_clusters=config['n_clusters'],
            random_state=config['random_state']
        )
        agg_df['region'] = kmeans.fit_predict(X_pca)
        agg_df['pca_comp1'] = X_pca[:, 0]
        agg_df['pca_comp2'] = X_pca[:, 1]
        
        # Simpan hasil
        output_csv = os.path.join(output_dir, 'regionalisasi_stasiun.csv')
        agg_df.to_csv(output_csv, index=False)
        print(f"✅ Hasil disimpan di: {output_csv}")
        
        # Visualisasi
        plt.figure(figsize=(8, 6))
        colors = plt.cm.tab10(np.linspace(0, 1, config['n_clusters']))
        for i in range(config['n_clusters']):
            cluster_mask = agg_df['region'] == i
            plt.scatter(
                agg_df.loc[cluster_mask, 'pca_comp1'],
                agg_df.loc[cluster_mask, 'pca_comp2'],
                label=f'Region {i}',
                color=colors[i],
                alpha=0.8,
                s=50
            )
        plt.title(f'Regionalisasi Stasiun ({param})\nBaseline {baseline} | {config["description"]}')
        plt.xlabel('PCA Component 1')
        plt.ylabel('PCA Component 2')
        plt.legend(title='Region')
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'plots', 'Station_Regions_PCA.png'), dpi=300, bbox_inches='tight')
        plt.close()
        
        print(f"✅ Selesai: {param} (baseline {baseline})")
        return True
        
    except Exception as e:
        print(f"❌ Gagal memproses {param} (baseline {baseline}): {e}")
        return False

# ==============================
# 3. EKSEKUSI UTAMA
# ==============================
if __name__ == "__main__":
    print("Memulai pipeline regionalisasi untuk kedua baseline (1981 dan 1991)...\n")
    
    summary = {}
    
    for baseline in BASELINES:
        print(f"\n{'#'*70}")
        print(f"# MEMPROSES BASELINE: {baseline}")
        print(f"{'#'*70}\n")
        
        for param, config in PARAM_CONFIG.items():
            success = process_parameter(param, config, baseline)
            key = f"{param}_{baseline}"
            summary[key] = "Berhasil" if success else "Gagal"
    
    # Ringkasan akhir
    print(f"\n{'='*60}")
    print("RINGKASAN EKSEKUSI")
    print(f"{'='*60}")
    for key, status in summary.items():
        print(f"{key:<40} : {status}")
    
    print(f"\n✅ Pipeline regionalisasi selesai.")
    print(f"Hasil tersedia di: {REGIONAL_DIR}/")

Memulai pipeline regionalisasi untuk kedua baseline (1981 dan 1991)...


######################################################################
# MEMPROSES BASELINE: 1981
######################################################################


Memproses: TEMPERATURE_AVG_C | Baseline: 1981
Deskripsi: Suhu rata-rata harian
Jumlah cluster: 4
Jumlah stasiun valid: 98
Komponen 1: Variansi = 0.8765, Kumulatif = 0.8765
Komponen 2: Variansi = 0.1130, Kumulatif = 0.9896
Komponen 3: Variansi = 0.0104, Kumulatif = 1.0000
✅ Hasil disimpan di: data/02.Regionalisasi_Dataset/TEMPERATURE_AVG_C_BASELINE_1981/regionalisasi_stasiun.csv
✅ Selesai: TEMPERATURE_AVG_C (baseline 1981)

Memproses: TEMP_24H_TN_C | Baseline: 1981
Deskripsi: Suhu minimum harian
Jumlah cluster: 4
Jumlah stasiun valid: 105
Komponen 1: Variansi = 0.8694, Kumulatif = 0.8694
Komponen 2: Variansi = 0.1160, Kumulatif = 0.9854
Komponen 3: Variansi = 0.0146, Kumulatif = 1.0000
✅ Hasil disimpan di: data/02.Regionalisasi_Dataset/TEMP_24H_TN